In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# Identifying Tautologies

In this notebook we develop a function `tautology` that takes a formula $f$ from propositional logic and checks whether $f$ is a *tautology*.
As tautologies are propositional formulas, we first have to import the parser for propositional logic.

In [ ]:
import_notebook "Propositional-Logic-Parser.ipynb"

To be able to compute all propositional valuations for a given formula $f$ we first need to determine the set of all variables that occur in $f$.  The function `collect_vars` takes one argument:
- `f` is a formula from propositional logic.

It returns the set of all propositional variables occurring in `f`.  The function is defined recursively: The variables of a formula like $g \wedge h$ are the variables of $g$ together with the variables of $h$.

**Implementation:** A pattern of the form `And (g, h) | Or (g, h) | Imp (g, h) | Iff (g, h)` is an *or-pattern*: It matches if one of its alternatives matches.  All alternatives have to bind the same variables, here `g` and `h`.

In [ ]:
let rec collect_vars (f : formula) : string RSet.t =
  match f with
  | Var p          -> RSet.singleton p
  | Verum | Falsum -> RSet.empty
  | Not g          -> collect_vars g
  | And (g, h) | Or (g, h) | Imp (g, h) | Iff (g, h) ->
      RSet.union (collect_vars g) (collect_vars h)

We have discussed the function <tt>evaluate</tt> previously.  The call
`evaluate f i` takes a propositional formula `f` and a propositional valuation `i`, where `i` is represented as a set of propositional variables.  It evaluates `f` given `i`.

In [ ]:
let rec evaluate (f : formula) (i : string RSet.t) : bool =
  match f with
  | Var p      -> RSet.mem p i
  | Verum      -> true
  | Falsum     -> false
  | Not g      -> not (evaluate g i)
  | And (g, h) -> evaluate g i && evaluate h i
  | Or  (g, h) -> evaluate g i || evaluate h i
  | Imp (g, h) -> evaluate g i <= evaluate h i
  | Iff (g, h) -> evaluate g i =  evaluate h i

Now we are ready to define the function `tautology`.  It takes one argument:
- `f` is a propositional formula.

The function checks whether `f` is a tautology.  In <samp>Python</samp>, this function could return either `True` or a counter example.  In <samp>OCaml</samp>, every function has a unique result type.  Therefore, we use the type `string RSet.t option`.  A value of this type has one of two forms:
- `None` indicates that `f` is a tautology,
- `Some i` indicates that `f` is not a tautology.  Then `i` is a set of variables such that $f$ evaluates to `false` if all variables in $i$ are `true`, while all variables not in $i$ are `false`.

**Implementation:** `List.find_opt p l` returns `Some x` for the first element `x` of the list `l` that satisfies the predicate `p`.  If there is no such element, it returns `None`.  This is exactly the result we need when we search for a valuation `i` that makes `f` false.  `RSet.to_list` converts the power set into a list.

In [ ]:
let tautology (f : formula) : string RSet.t option =
  let vars = collect_vars f in
  List.find_opt (fun i -> not (evaluate f i)) (RSet.to_list (RSet.powerset vars))

The function `test` takes one argument:
- `s` is a string that can be parsed as a propositional formula.

It checks whether this formula is a tautology and prints the result.  If the formula is not a tautology, a counter example is printed.

In [ ]:
let test (s : string) : unit =
  let f = parse s in
  match tautology f with
  | None ->
      Printf.printf "The formula %s is a tautology.\n" s
  | Some counter_example ->
      Printf.printf "The formula %s is not a tautology.\n" s;
      print_endline "Counter example:";
      RSet.iter
        (fun x -> Printf.printf "%s ↦ %b\n" x (RSet.mem x counter_example))
        (collect_vars f)

Let us run a few tests.

Let's check the rules of *DeMorgan*.

In [ ]:
test "¬(p ∨ q) ↔ ¬p ∧ ¬q"

In [ ]:
test "¬(p ∧ q) ↔ ¬p ∨ ¬q"

In [ ]:
test "(p → q) → (¬p → q) → q"

In [ ]:
test "(p → q) → (¬p → ¬q)"

In [ ]:
test "(p → q) → (¬q → ¬p)"

In [ ]:
test "¬p ↔ (p → ⊥)"